*  DSC 530
*  Final
*  Saebastion Cole

# FIFA 2026 Player Performance

## Setup

In [162]:
# Begin by adding all of the needed imports at the top
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random as random

In [163]:
# Read in the player stats data

player_stats = pd.read_csv("data/fifa_world_cup_2026_player_performance.csv")

player_stats.head()

,player_id,player_name,age,nationality,team,jersey_number,position,height_cm,weight_kg,preferred_foot,...,possession_impact,pressure_resistance,creativity_score,consistency_score,clutch_performance_score,total_goals_tournament,total_assists_tournament,total_minutes_tournament,player_of_match_awards,tournament_rating
0,P00055,Rodri Fati,26,Spanish,Spain,3,Goalkeeper,195,75,Left,...,1.1,44.2,55.9,42.0,51.8,0,0,242,0,5.8
1,P00070,Ansu Le Normand,19,Spanish,Spain,18,Midfielder,178,75,Right,...,3.5,38.2,43.7,31.1,52.7,0,3,342,0,5.5
2,P00066,Gavi Ramos,18,Spanish,Spain,14,Midfielder,177,72,Left,...,15.3,99.0,99.0,83.4,54.8,1,1,245,0,8.4
3,P00073,Pedro Cubarsi,20,Spanish,Spain,21,Forward,182,74,Right,...,1.2,19.8,42.3,40.9,78.5,5,3,422,0,6.7
4,P00059,Alvaro Oyarzabal,23,Spanish,Spain,7,Defender,191,81,Left,...,6.2,44.1,33.5,60.0,56.6,0,0,440,0,5.7


## Cleaning

Not much cleaning is required with this dataset since it already has been cleaned and seems complete, I reviewed and didn't see any duplicates or anything.

So since players wont be switching national teams we can assume if they play one game for one then they are playing for that team for all other games.

#### Convert Match Date to Object


In [164]:
# Convert the match_date column to a datetime object
player_stats["match_date_obj"] = pd.to_datetime(player_stats["match_date"], format="%Y-%m-%d")

# Sort the dataset
player_stats = player_stats.sort_values(by="match_date_obj")

player_stats["match_date_obj"].head()

6317    2026-06-11
23907   2026-06-11
23906   2026-06-11
23905   2026-06-11
23904   2026-06-11
Name: match_date_obj, dtype: datetime64[us]

#### Set the index
Index is player and match IDs combined since no player should have multiple records for a match.

In [165]:
# Create an index for the pair to help with filtering speed
player_stats.set_index(["player_name", "match_id"], inplace=True)
player_stats.head()

,,player_id,age,nationality,team,jersey_number,position,height_cm,weight_kg,preferred_foot,club_name,...,pressure_resistance,creativity_score,consistency_score,clutch_performance_score,total_goals_tournament,total_assists_tournament,total_minutes_tournament,player_of_match_awards,tournament_rating,match_date_obj
player_name,match_id,,,,,,,,,,,,,,,,,,,,,
Giovanni Weah,M00122,P00653,33,American,United States,3,Goalkeeper,189,77,Right,LAFC,...,46.5,47.5,39.3,72.1,0,0,118,0,0.0,2026-06-11
Selim Saiss,M00460,P01060,21,Moroccan,Morocco,20,Midfielder,186,82,Right,Al-Hilal Omdurman,...,96.3,60.3,99.0,43.5,0,0,88,0,7.7,2026-06-11
Nayef Saibari,M00460,P01055,24,Moroccan,Morocco,15,Midfielder,175,70,Right,ASEC Mimosas,...,51.9,44.8,50.4,50.7,0,0,150,0,5.7,2026-06-11
Noussair El Khannouss,M00460,P01050,23,Moroccan,Morocco,10,Defender,180,74,Left,Club Africain,...,38.2,44.4,63.7,71.3,0,0,257,0,5.5,2026-06-11
Achraf Amellah,M00460,P01053,24,Moroccan,Morocco,13,Midfielder,186,77,Left,JS Kabylie,...,46.6,40.8,68.3,62.7,0,3,354,0,5.8,2026-06-11


#### Remove players that did not participate
For this the data will be cleaned picking only the players who have more than 0 minutes in.

In [166]:
# Get data for players who actually competed
player_stats_active = player_stats[player_stats["minutes_played"] > 0]

player_stats_active.head()

,,player_id,age,nationality,team,jersey_number,position,height_cm,weight_kg,preferred_foot,club_name,...,pressure_resistance,creativity_score,consistency_score,clutch_performance_score,total_goals_tournament,total_assists_tournament,total_minutes_tournament,player_of_match_awards,tournament_rating,match_date_obj
player_name,match_id,,,,,,,,,,,,,,,,,,,,,
Selim Saiss,M00460,P01060,21,Moroccan,Morocco,20,Midfielder,186,82,Right,Al-Hilal Omdurman,...,96.3,60.3,99.0,43.5,0,0,88,0,7.7,2026-06-11
Nayef Saibari,M00460,P01055,24,Moroccan,Morocco,15,Midfielder,175,70,Right,ASEC Mimosas,...,51.9,44.8,50.4,50.7,0,0,150,0,5.7,2026-06-11
Noussair El Khannouss,M00460,P01050,23,Moroccan,Morocco,10,Defender,180,74,Left,Club Africain,...,38.2,44.4,63.7,71.3,0,0,257,0,5.5,2026-06-11
Achraf Amellah,M00460,P01053,24,Moroccan,Morocco,13,Midfielder,186,77,Left,JS Kabylie,...,46.6,40.8,68.3,62.7,0,3,354,0,5.8,2026-06-11
Romain Aboukhlal,M00460,P01054,18,Moroccan,Morocco,14,Midfielder,183,85,Right,TP Mazembe,...,67.0,99.0,83.4,71.0,2,1,456,0,6.0,2026-06-11


### Sampling
The dataset contains the entire dataset of all players. Since this data is pretty detailed and contains everything we need this isn't really required. Instead of random sampling I will just use the actual data since there are plenty of rows to work with I won't be using this in my actual analysis since the dataset is small enough to not slow down calculations.

In [ ]:
# Take a 10% sample of the player-match records. This makes the dataset much smaller for testing.
player_stats_sample = (
    player_stats_active
    .groupby("position", group_keys=False)
    .sample(frac=0.10, random_state=0)
    .sort_values(by="match_date_obj")
)


player_stats_sample.head()

,,player_id,age,nationality,team,jersey_number,position,height_cm,weight_kg,preferred_foot,club_name,...,pressure_resistance,creativity_score,consistency_score,clutch_performance_score,total_goals_tournament,total_assists_tournament,total_minutes_tournament,player_of_match_awards,tournament_rating,match_date_obj
player_name,match_id,,,,,,,,,,,,,,,,,,,,,
Thomas Openda,M00739,P00167,35,Belgian,Belgium,11,Defender,189,79,Right,Lech Poznan,...,56.0,30.7,57.2,43.2,0,1,297,0,6.0,2026-06-11
Nabil Bennacer,M00617,P01190,24,Algerian,Algeria,20,Midfielder,178,76,Left,Raja Casablanca,...,43.8,55.5,43.6,33.7,0,2,427,0,5.7,2026-06-11
Ehsan Hosseini,M00908,P00884,25,Iranian,Iran,26,Forward,170,74,Right,Pakhtakor,...,74.9,69.2,80.5,71.0,1,1,347,0,0.0,2026-06-11
Viktor Krafth,M00654,P00371,23,Swedish,Sweden,7,Defender,183,79,Right,Paris Saint-Germain,...,39.5,40.4,46.4,52.5,0,0,40,0,5.2,2026-06-11
Ciro Colpani,M00076,P00122,26,Italian,Italy,18,Midfielder,177,76,Right,Rapid Vienna,...,97.2,71.4,99.0,59.4,0,1,361,0,7.1,2026-06-11


## Analysis

### Calculate Statistics for all data

#### Mean

In [168]:
player_stats["player_rating"].mean()

np.float64(3.634699633699634)

In [169]:
player_stats["successful_passes"].mean()

np.float64(15.470091575091574)

#### Median

In [170]:
player_ratings.median()

np.float64(5.5)

In [171]:
player_stats["successful_passes"].median()

np.float64(9.0)

#### Mode

In [172]:
vals, counts = np.unique(player_stats["player_rating"], return_counts=True)
# Return the value with the highest count and the counts 
mode = vals[counts.argmax()], counts.max()
mode

(np.float64(0.0), np.int64(23042))

### Calculating Statistics for players with time


#### Mean

In [173]:
player_stats_active["player_rating"].mean()

np.float64(6.288567082831612)

#### Median

In [174]:
player_stats_active["player_rating"].median()

np.float64(6.3)

#### Comparison
Overall removing the players who had not actually participated in the games pretty significantly effects the mean and much less the median as expected. Mode can be left out of this consideration since nearly half of the players in the event do not actually get play time.

- Player Ratings

    The mean nearly doubled after taking out the players who did not participate to be expected. This made the median and mean very closely align with players in game getting an average ~6.
    
- Successful Passes
    
    It is noticable that the mean is much higher than the median pointing out that key players had significantly higher successful pass counts than an average.

- Other

    Other analysis on the overall dataset may not help much like seeing defensive stats because a defensive player should have a much higher stat similar to a attacking role having more shots on goal.